# 04-5. CSV 읽기·검증·안전한 출력 실습

## Goal

표준 csv 모듈로 따옴표·쉼표·줄바꿈을 올바르게 처리하고, 헤더·행 구조·자료형·범위를 검증한 뒤 안전한 출력 파일을 만듭니다. 각 단계는 **결과 예측 → 실행 → 이유 설명 → 입력 변경** 순서로 진행하세요.


## Setup

모든 CSV 파일은 임시 디렉터리에 직접 생성합니다. 실제 업무 파일이나 스프레드시트 파일을 열지 않으며 Python 3.10 이상의 표준 라이브러리만 사용합니다.


In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory
import csv

csv_tempdir = TemporaryDirectory(prefix="python-04-5-")
lab_dir = Path(csv_tempdir.name)
assert lab_dir.is_dir()


def expect_exception(exception_type, function, *args, **kwargs):
    try:
        function(*args, **kwargs)
    except exception_type as exc:
        return exc
    raise AssertionError(f"{exception_type.__name__}이 발생해야 합니다")


print("격리 실습 디렉터리:", lab_dir)


## Steps

### 1. csv.writer로 재현 가능한 입력 만들기

split(",")은 따옴표 안의 쉼표나 줄바꿈을 이해하지 못합니다. writer가 CSV 규칙에 맞는 입력을 만들게 합니다.


In [ ]:
people_path = lab_dir / "people.csv"
people_rows = [
    ["name", "age", "city"],
    ["Alice", "20", "Seoul"],
    ["Bob", "22", "Suwon, Gyeonggi"],
    ["Carol", "30", "Busan\nHaeundae"],
]

with people_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    writer = csv.writer(file)
    writer.writerows(people_rows)

assert people_path.is_file()
assert '"Suwon, Gyeonggi"' in people_path.read_text(encoding="utf-8")
print(people_path.read_text(encoding="utf-8"))


### 2. reader와 DictReader 비교

reader는 문자열 리스트를, DictReader는 헤더 이름을 키로 사용하는 딕셔너리를 반환합니다. 숫자처럼 보이는 필드도 먼저 문자열로 읽힙니다.


In [ ]:
with people_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    reader_rows = list(csv.reader(file))

with people_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    dictionary_reader = csv.DictReader(file)
    fieldnames = dictionary_reader.fieldnames
    people_as_dicts = list(dictionary_reader)

assert reader_rows[2][2] == "Suwon, Gyeonggi"
assert reader_rows[3][2] == "Busan\nHaeundae"
assert fieldnames == ["name", "age", "city"]
assert people_as_dicts[0]["age"] == "20"
assert isinstance(people_as_dicts[0]["age"], str)

print(people_as_dicts)


### 3. 입력 계약을 바꾸어 보기

구분자가 세미콜론인 입력은 delimiter를 명시해야 합니다. delimiter 값을 쉼표로 바꾸면 결과가 왜 달라지는지 예상해 보세요.


In [ ]:
semicolon_rows = list(csv.reader(
    ["name;age;city", "Alice;20;Seoul"],
    delimiter=";",
))

assert semicolon_rows == [
    ["name", "age", "city"],
    ["Alice", "20", "Seoul"],
]
print(semicolon_rows)


### 4. 헤더와 레코드 구조 검증

헤더 검증을 자료형 변환보다 먼저 수행합니다. DictReader에서 초과 필드는 None 키에, 누락 필드는 값 None으로 나타나며 빈 셀 ""과는 구분됩니다.


In [ ]:
REQUIRED_FIELDS = {"name", "age", "city"}
ALLOWED_FIELDS = {"name", "age", "city"}


def validate_headers(fieldnames):
    if fieldnames is None:
        raise ValueError("CSV 헤더가 없습니다")
    if any(name is None or not name.strip() for name in fieldnames):
        raise ValueError("비어 있는 CSV 헤더가 있습니다")
    if len(fieldnames) != len(set(fieldnames)):
        raise ValueError("중복 CSV 헤더가 있습니다")

    names = set(fieldnames)
    missing = REQUIRED_FIELDS - names
    unknown = names - ALLOWED_FIELDS

    if missing:
        raise ValueError(f"필수 헤더 누락: {sorted(missing)}")
    if unknown:
        raise ValueError(f"알 수 없는 헤더: {sorted(unknown)}")


def parse_person(row):
    if None in row:
        raise ValueError("헤더보다 값이 많은 레코드입니다")
    if any(value is None for value in row.values()):
        raise ValueError("필드가 누락된 레코드입니다")

    name = row["name"].strip()
    age_text = row["age"].strip()
    city = row["city"].strip()

    if not name or not age_text or not city:
        raise ValueError("빈 필드가 있습니다")

    try:
        age = int(age_text)
    except ValueError as exc:
        raise ValueError("나이는 정수여야 합니다") from exc

    if not 0 <= age <= 130:
        raise ValueError("나이는 0부터 130 사이여야 합니다")

    return {"name": name, "age": age, "city": city}


assert validate_headers(["name", "age", "city"]) is None
assert "중복" in str(expect_exception(
    ValueError, validate_headers, ["name", "age", "age"]
))
assert "비어" in str(expect_exception(
    ValueError, validate_headers, ["name", "", "city"]
))
assert "알 수 없는" in str(expect_exception(
    ValueError, validate_headers, ["name", "age", "city", "token"]
))


### 5. 오류의 레코드 번호와 물리 행 번호 보존

따옴표 안의 줄바꿈 때문에 논리 레코드 번호와 물리 행 번호가 달라질 수 있습니다. 오류 기록에는 위치와 일반화한 원인만 남기고 입력 전문은 복사하지 않습니다.


In [ ]:
people_with_error_path = lab_dir / "people-with-error.csv"
people_with_error_rows = [
    ["name", "age", "city"],
    ["Alice", "20", "Seoul"],
    ["Bob", "22", "Suwon\nGyeonggi"],
    ["Carol", "unknown", "Busan"],
]

with people_with_error_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    csv.writer(file).writerows(people_with_error_rows)


def load_people(path):
    valid_records = []
    errors = []

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as file:
        reader = csv.DictReader(
            file,
            delimiter=",",
            quotechar='"',
            strict=True,
        )
        try:
            validate_headers(reader.fieldnames)

            for record_number, row in enumerate(reader, start=1):
                try:
                    record = parse_person(row)
                except (KeyError, TypeError, ValueError) as exc:
                    errors.append({
                        "record": record_number,
                        "line": reader.line_num,
                        "error": str(exc),
                    })
                    continue
                valid_records.append(record)
        except csv.Error as exc:
            raise ValueError(
                f"CSV 문법 오류(물리 행 {reader.line_num}): {exc}"
            ) from exc

    return valid_records, errors


valid_people, people_errors = load_people(people_with_error_path)

assert len(valid_people) == 2
assert valid_people[1]["city"] == "Suwon\nGyeonggi"
assert people_errors == [{
    "record": 3,
    "line": 5,
    "error": "나이는 정수여야 합니다",
}]
assert set(people_errors[0]) == {"record", "line", "error"}

print("정상:", valid_people)
print("오류:", people_errors)


### 6. DictWriter·newline=""·출력 스키마

출력 열과 순서를 고정하고, 쓰기 전에 각 레코드가 정확한 키를 가졌는지 확인합니다. 다시 읽어 헤더와 건수를 검증해야 저장 성공을 확인할 수 있습니다.


In [ ]:
OUTPUT_FIELDS = ["name", "age", "city"]


def write_people(path, records, *, encoding="utf-8"):
    expected_keys = set(OUTPUT_FIELDS)
    for record in records:
        if set(record) != expected_keys:
            raise ValueError("출력 레코드 키 구성이 올바르지 않습니다")

    with path.open(
        "w",
        encoding=encoding,
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=OUTPUT_FIELDS,
            extrasaction="raise",
        )
        writer.writeheader()
        writer.writerows(records)


valid_people_path = lab_dir / "people-valid.csv"
write_people(valid_people_path, valid_people)

with valid_people_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    restored_people = list(csv.DictReader(file))

assert len(restored_people) == len(valid_people)
assert restored_people[0]["name"] == "Alice"
assert restored_people[0]["age"] == "20"
assert "키 구성" in str(expect_exception(
    ValueError,
    write_people,
    lab_dir / "should-not-exist.csv",
    [{"name": "Only name"}],
))

print(restored_people)


### 7. UTF-8 BOM과 스프레드시트 수식 정책

BOM은 교환 계약이 요구할 때만 utf-8-sig로 사용합니다. CSV 따옴표는 스프레드시트의 수식 해석을 막지 않으므로 수식이 필요 없는 텍스트 열에는 별도 정책을 적용합니다.


In [ ]:
excel_path = lab_dir / "people-excel.csv"
write_people(excel_path, valid_people, encoding="utf-8-sig")

FORMULA_PREFIXES = ("=", "+", "-", "@")


def validate_spreadsheet_text(value):
    if value.lstrip().startswith(FORMULA_PREFIXES):
        raise ValueError(
            "스프레드시트 수식으로 해석될 수 있는 텍스트입니다"
        )
    return value


assert excel_path.read_bytes().startswith(b"\xef\xbb\xbf")
with excel_path.open(
    "r",
    encoding="utf-8-sig",
    newline="",
) as file:
    assert csv.DictReader(file).fieldnames == OUTPUT_FIELDS

assert validate_spreadsheet_text("Alice") == "Alice"
formula_error = expect_exception(
    ValueError, validate_spreadsheet_text, "  =2+3"
)
assert "수식" in str(formula_error)

print("BOM과 수식 위험 정책을 확인했습니다.")


## Checks

### 8. 상품 CSV 검증기

정상 상품, 쉼표가 든 상품명, 빈 값, 잘못된 정수, 음수, 누락·초과 필드, 수식 위험을 한 파일에 넣습니다. 정상·오류·격리 건수의 합이 전체 입력 수와 같은지 확인합니다.


In [ ]:
PRODUCT_FIELDS = {"name", "price", "quantity"}


class FormulaRiskError(ValueError):
    pass


def validate_product_headers(fieldnames):
    if fieldnames is None:
        raise ValueError("CSV 헤더가 없습니다")
    if len(fieldnames) != len(set(fieldnames)):
        raise ValueError("중복 CSV 헤더가 있습니다")
    if set(fieldnames) != PRODUCT_FIELDS:
        raise ValueError("상품 CSV 헤더 구성이 올바르지 않습니다")


def parse_product(row):
    if None in row:
        raise ValueError("헤더보다 값이 많은 레코드입니다")
    if any(value is None for value in row.values()):
        raise ValueError("필드가 누락된 레코드입니다")

    name = row["name"].strip()
    price_text = row["price"].strip()
    quantity_text = row["quantity"].strip()

    if not name or not price_text or not quantity_text:
        raise ValueError("빈 필드가 있습니다")
    try:
        validate_spreadsheet_text(name)
    except ValueError as exc:
        raise FormulaRiskError(str(exc)) from exc

    try:
        price = int(price_text)
        quantity = int(quantity_text)
    except ValueError as exc:
        raise ValueError("가격과 수량은 정수여야 합니다") from exc

    if price < 0:
        raise ValueError("가격은 0 이상이어야 합니다")
    if quantity < 0:
        raise ValueError("수량은 0 이상이어야 합니다")

    return {
        "name": name,
        "price": price,
        "quantity": quantity,
        "total": price * quantity,
    }


def load_products(path):
    valid_records = []
    errors = []
    quarantined = []
    total = 0

    with path.open(
        "r",
        encoding="utf-8",
        newline="",
    ) as file:
        reader = csv.DictReader(file, strict=True)
        validate_product_headers(reader.fieldnames)

        for record_number, row in enumerate(reader, start=1):
            total += 1
            position = {
                "record": record_number,
                "line": reader.line_num,
            }
            try:
                product = parse_product(row)
            except FormulaRiskError as exc:
                quarantined.append({
                    **position,
                    "error": str(exc),
                })
            except (KeyError, TypeError, ValueError) as exc:
                errors.append({
                    **position,
                    "error": str(exc),
                })
            else:
                valid_records.append(product)

    return {
        "total": total,
        "records": valid_records,
        "errors": errors,
        "quarantined": quarantined,
    }


In [ ]:
products_path = lab_dir / "products.csv"
product_rows = [
    ["name", "price", "quantity"],
    ["Notebook", "1000", "2"],
    ["USB, Cable", "500", "3"],
    ["", "100", "1"],
    ["Free sample", "free", "1"],
    ["Damaged box", "200", "-1"],
    ["Mouse", "300"],
    ["Hub", "500", "2", "extra"],
    ["=2+3", "100", "1"],
]

with products_path.open(
    "w",
    encoding="utf-8",
    newline="",
) as file:
    csv.writer(file).writerows(product_rows)

product_result = load_products(products_path)

assert product_result["total"] == 8
assert len(product_result["records"]) == 2
assert len(product_result["errors"]) == 5
assert len(product_result["quarantined"]) == 1
assert (
    len(product_result["records"])
    + len(product_result["errors"])
    + len(product_result["quarantined"])
    == product_result["total"]
)
assert product_result["records"][0]["total"] == 2000
assert product_result["records"][1]["name"] == "USB, Cable"
assert all(
    set(item) == {"record", "line", "error"}
    for item in product_result["errors"] + product_result["quarantined"]
)

print({
    "total": product_result["total"],
    "valid": len(product_result["records"]),
    "errors": len(product_result["errors"]),
    "quarantined": len(product_result["quarantined"]),
})


### 9. 정상 결과를 새 CSV로 저장하고 다시 읽기

오류와 격리 레코드는 출력에서 제외하고, 정상 레코드만 고정된 스키마로 저장합니다.


In [ ]:
PRODUCT_OUTPUT_FIELDS = ["name", "price", "quantity", "total"]


def write_valid_products(path, records):
    expected_keys = set(PRODUCT_OUTPUT_FIELDS)
    for record in records:
        if set(record) != expected_keys:
            raise ValueError("상품 출력 키 구성이 올바르지 않습니다")

    with path.open(
        "w",
        encoding="utf-8",
        newline="",
    ) as file:
        writer = csv.DictWriter(
            file,
            fieldnames=PRODUCT_OUTPUT_FIELDS,
            extrasaction="raise",
        )
        writer.writeheader()
        writer.writerows(records)


products_output_path = lab_dir / "products-valid.csv"
write_valid_products(
    products_output_path,
    product_result["records"],
)

with products_output_path.open(
    "r",
    encoding="utf-8",
    newline="",
) as file:
    output_reader = csv.DictReader(file)
    output_header = output_reader.fieldnames
    restored_products = list(output_reader)

assert output_header == PRODUCT_OUTPUT_FIELDS
assert len(restored_products) == 2
assert restored_products[0]["total"] == "2000"
assert restored_products[1]["name"] == "USB, Cable"

print(restored_products)


In [ ]:
final_checks = {
    "CSV 규칙으로 쉼표 처리": reader_rows[2][2] == "Suwon, Gyeonggi",
    "필드는 문자열로 입력": isinstance(people_as_dicts[0]["age"], str),
    "물리 행 위치 보존": people_errors[0]["line"] == 5,
    "BOM 계약": excel_path.read_bytes().startswith(b"\xef\xbb\xbf"),
    "수식 위험 격리": len(product_result["quarantined"]) == 1,
    "처리 건수 보존": (
        len(product_result["records"])
        + len(product_result["errors"])
        + len(product_result["quarantined"])
        == product_result["total"]
    ),
    "출력 재검증": len(restored_products) == len(product_result["records"]),
}

for name, passed in final_checks.items():
    assert passed
    print(f"[PASS] {name}")


## Next Steps

- CSV 파싱과 업무 스키마 검증을 별도 단계로 설명합니다.
- 레코드 번호와 물리 행 번호를 구분하고 오류에 입력 전문을 남기지 않습니다.
- 출력에는 newline=""과 고정 fieldnames를 사용하고 다시 읽어 검증합니다.
- 다음 절에서는 같은 검증 결과를 JSON 문서와 JSON Lines로 직렬화합니다.


In [ ]:
csv_tempdir.cleanup()
assert not lab_dir.exists()
print("임시 실습 디렉터리를 정리했습니다.")
